# Grid Topology Generation

This notebook demonstrates how to artificially generate a power grid topology using the Chung-Lu-Chain graph model.

In [ ]:
# ── Colab-only setup (run once, then restart the runtime) ──
# Colab pre-loads NumPy's C extensions into memory. Installing pandapower may
# upgrade NumPy, but the old .so files stay loaded and cause ImportError.
# Pinning versions + restarting the runtime fixes this.
%pip install -q "numpy>=2.2,<3" "scipy>=1.15,<2" "pandapower>=3.3,<4"
%pip install -q git+https://github.com/cookbook-ms/Chung_Lu_Chain-synthesizer.git@main

# Restart the Colab runtime so the new NumPy C extensions are loaded.
# After the restart, skip this cell and run the next one.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## Basics

In [ ]:
from powergrid_synth import (
    PowerGridGenerator,
    InputConfigurator,
    HierarchicalAnalyzer,
    GridVisualizer,
)

## CLC topology generation
We consider a Chung-Lu-Chain graph model for the grid topology generation. It contains two phases:

- **Phase 1**: For each the same-voltage subgraph, we consider the Chung-Lu-Chain model that takes as input __the desired node degree sequence__ and __the desired graph diameter__. 

- **Phase 2**: After the same-voltage subgraphs are created, transformer edges between each pair of subgraphs are inserted. This takes as input **the desired transformer degree of each vertex in the subgraphs of voltage** $X$ and $Y$. 

### Two operation modes
1. _Mode I:_ When using the CLC graph generation model, one requires __desired same-voltage subgraph degrees and diameter__ for Phase 1, and __desired transformer degrees__ for Phase 2. They can be easily extracted from given (realistic) power grid topology. We can refer to this as __Operation mode I__ --- one can fit the model to existing power grid graphs and create ensembles of structurally similar graphs. 

2. _Mode II:_ In cases where real-world grid data is completely unavailable or when users want to scale or vary the inputs, we would prefer to use _ONLY_ artificial input. This __Opertaion mode II__ requires users only specifying the number of vertices in each same-voltage subgraph. Then, one would need to generate the artificial degree sequences and transformer degrees in some way, e.g., statistically using some __degree distributions__ or __fitting functions__.
    - In this mode, users are free to decide which assumptions are most appropriate for their purposes. 
    - In this project, we consider some statistically-significant fitting functions and distributions to generate the needed input. 

### Inputs for the degree distributions and transformer connection distributions

In [ ]:
# Define 3 voltage levels
level_specs = [
    # Level 0
    {'n': 20, 'avg_k': 4.0, 'diam': 6, 'dist_type': 'dgln'},
    # Level 1
    {'n': 20, 'avg_k': 3.0, 'diam': 10, 'dist_type': 'dpl'},
    # Level 2
    {'n': 10, 'avg_k': 2.0, 'diam': 10, 'dist_type': 'dgln'}
]

connection_specs = {
    (0, 1): {'type': 'k-stars', 'c': 0.174, 'gamma': 4.15},
    (1, 2): {'type': 'k-stars', 'c': 0.15, 'gamma': 4.15}
}

### Define an input configurator
It configurates the input parameters to the needed ones for the algorithm

In [ ]:
print("\n[1] Configuring 3-Level Hierarchy...")

# Initialize Configurator
configurator = InputConfigurator(seed=100)

# Generating Input Parameters
params = configurator.create_params(level_specs, connection_specs)

### Generate the grid topology

In [ ]:
gen = PowerGridGenerator(seed=100)

print("\n[2] Generating Topology...")

grid_graph = gen.generate_grid(
    degrees_by_level=params['degrees_by_level'],
    diameters_by_level=params['diameters_by_level'],
    transformer_degrees=params['transformer_degrees'], 
    keep_lcc=True,
)
print(f"Grid Generated: {grid_graph.number_of_nodes()} nodes, {grid_graph.number_of_edges()} edges")

### Visualize the generated grid

In [ ]:
viz = GridVisualizer()

In [ ]:
print("\n[3] Visualizing Full Grid Topology...")
viz.plot_grid(
    grid_graph, 
    layout='kamada_kawai',
    title="3-Level Synthetic Grid",
    figsize=(6, 4)
)

In [ ]:
sub_lv0 = grid_graph.level(0)
sub_lv1 = grid_graph.level(1)

In [ ]:
print("\n[3] Visualizing Sub-Grid Topology...")
viz.plot_subgraphs(grid_graph, figsize=(10,3))

### Analyze the generated topology

In [ ]:
print("\n[4] Running Hierarchical Analysis...")

# Initialize the Analyzer
analyzer = HierarchicalAnalyzer(grid_graph)

# Run the full report
analyzer.run_full_report(log_scale=True)

print("Analysis Complete.")

### Prescribed vs synthetic degree distribution

The CLC model takes a *prescribed* (desired) degree sequence as input and produces a random graph whose *actual* degree sequence approximates it. Here we compare the two per voltage level using the **KS statistic** and the **Relative Hausdorff (RH) distance**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from scipy.stats import ks_2samp
from scipy.spatial.distance import directed_hausdorff

def relative_hausdorff(seq_a, seq_b):
    """1-D Relative Hausdorff distance between two degree sequences."""
    a = np.sort(seq_a).reshape(-1, 1).astype(float)
    b = np.sort(seq_b).reshape(-1, 1).astype(float)
    h = max(directed_hausdorff(a, b)[0], directed_hausdorff(b, a)[0])
    denom = max(a.max(), b.max())
    return h / denom if denom > 0 else 0.0

rows = []
for level, prescribed_degrees in enumerate(params['degrees_by_level']):
    sub = grid_graph.level(level)
    actual_degrees = [d for _, d in sub.degree()]

    ks_stat, _ = ks_2samp(prescribed_degrees, actual_degrees)
    rh = relative_hausdorff(prescribed_degrees, actual_degrees)

    rows.append({
        'Level': f'Level {level}',
        'Prescribed #nodes': len(prescribed_degrees),
        'Actual #nodes': len(actual_degrees),
        'KS Statistic': round(ks_stat, 4),
        'RH Distance': round(rh, 4),
    })

df = pd.DataFrame(rows)
print("Prescribed vs Synthetic Degree Distribution Comparison")
df

In [ ]:
n_levels = len(params['degrees_by_level'])
fig, axes = plt.subplots(1, n_levels, figsize=(5 * n_levels, 4))
if n_levels == 1:
    axes = [axes]

for level, ax in enumerate(axes):
    prescribed = params['degrees_by_level'][level]
    actual = [d for _, d in grid_graph.level(level).degree()]

    c_pre = Counter(prescribed)
    c_act = Counter(actual)
    all_k = sorted(set(list(c_pre.keys()) + list(c_act.keys())))

    p_pre = [c_pre.get(k, 0) / len(prescribed) for k in all_k]
    p_act = [c_act.get(k, 0) / len(actual) for k in all_k]

    width = 0.35
    x = np.arange(len(all_k))
    ax.bar(x - width/2, p_pre, width, label='Prescribed', color='orange', alpha=0.8)
    ax.bar(x + width/2, p_act, width, label='Synthetic', color='blue', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(all_k)
    ax.set_xlabel('Degree')
    ax.set_ylabel('Probability')
    ax.set_title(f'Level {level}')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Prescribed vs Synthetic Degree Distributions', y=1.02)
plt.tight_layout()
plt.show()